# Development 6: the C++ geometry core, tested against the Python reference

Builds `cpp/` on this runtime, runs the differential tests, then checks
agreement and speed on REAL frames. Python stays the reference: if the two ever
disagree, the C++ is wrong.

**CPU server.** Cells 1 to 5 need no data and take about a minute. Cell 6
downloads the block with LiDAR (about 8 min) for the real-data check and the
per-step timing; set `USE_REAL_DATA = False` to skip it.

**Read the saved output with this in mind:** it was produced with the FIRST version of the C++ occlusion test, which
was slower than NumPy on real frames (51 ms against 31 ms; whole scene 1.4x faster in C++). That result is why the
test was rewritten afterwards (each point scans its own pixel row and column and stops at the first hit). The
rewritten version, which the pipeline uses, was timed on a laptop only: `docs/performance.md`. Agreement between
C++ and Python was exact before and after.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
USE_REAL_DATA = True
SPLIT, BLOCK = "val", 11
LAYERS = ["camera_keyframes", "lidar_motion_compensated_keyframes"]
SCENE_ID = "2025-06-13-07-09-37|78"
CAMERA = "front_medium"

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Session, then build the C++ module on this machine ---
import subprocess, sys, time
from vggt_aura.session import start_session
from vggt_aura import cpp_build

session = start_session(persist_mode=PERSIST_MODE, require_gpu=False)
for tool in ("g++ --version", "cmake --version"):
    result = subprocess.run(tool, shell=True, capture_output=True, text=True)
    print(tool.split()[0], "->", result.stdout.splitlines()[0] if result.returncode == 0 else "MISSING")
started = time.time()
BUILT = cpp_build.build()
print(f"build {'succeeded' if BUILT else 'FAILED'} in {time.time() - started:.1f} s")
cpp = cpp_build.import_module()
print("module:", getattr(cpp, "__file__", None))
print("functions:", [name for name in dir(cpp) if not name.startswith("_")] if cpp else None)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
g++ -> g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
cmake -> cmake version 3.31.10
$ cmake -S /content/vggt-omega-aura-benchmark/cpp -B /content/vggt-omega-aura-benchmark/cpp/build -DCMAKE_BUILD_TYPE=Release -Dpybind11_DIR=/usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11 -DPython_EXECUTABLE=/usr/bin/python3
$ cmake --build /content/vggt-omega-aura-benchmark/cpp/build --config Release -j
build succeeded in 15.6 s
module: /content/vggt-omega-aura-benchmark/cpp/build/vggt_geom_cpp.cpython-313-x86_64-linux-gnu.so
functions: ['compose', 'invert_se3', 'lidar_to_image', 'nearest_per_pixel', 'occlusion_reason', 'pixel_indices', 'project_pinhole', 'quat_xyzw_to_matrix', 'transform_points', 'unproject_pinhole']


In [4]:
# --- 4. The differential test suite, run HERE against the module just built ---
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                         str(PROJECT_DIR / "tests" / "test_cpp_differential.py")],
                        capture_output=True, text=True, cwd=str(PROJECT_DIR))
print(result.stdout[-2500:])
print(result.stderr[-800:])

..........................                                               [100%]
26 passed in 2.93s




In [5]:
# --- 5. Speed on synthetic frames of realistic size (no data needed). Best of 5 runs each ---
import numpy as np
import pandas as pd
from vggt_aura import ground_truth as gtm

pd.set_option("display.width", 200)
W, H, K = 640, 400, (538.2, 564.73, 324.75, 196.27)
P = gtm.OcclusionParams()


def best_ms(fn, repeats=5):
    times = []
    for _ in range(repeats):
        started = time.perf_counter()
        fn()
        times.append(time.perf_counter() - started)
    return min(times) * 1000.0


def stage_table(points, camera_from_source):
    rotation, translation = camera_from_source[:3, :3], camera_from_source[:3, 3]
    cam = points @ rotation.T + translation
    u, v, z = gtm.project_pinhole(cam, *K)
    ui, vi, inside = gtm.pixel_indices(u, v, W, H)
    keep = inside & (z >= P.min_depth)
    cu, cv, cz = ui[keep], vi[keep], z[keep]
    loaded = {"points_cam": cam, "semantic_id": np.ones(len(cam), np.uint16),
              "instance_id": np.zeros(len(cam), np.uint16), "sensor_index": np.zeros(len(cam), np.uint8)}
    rows = [
        ("transform points", lambda: points @ rotation.T + translation, lambda: cpp.transform_points(camera_from_source, points)),
        ("project", lambda: gtm.project_pinhole(cam, *K), lambda: cpp.project_pinhole(cam, *K)),
        ("pixel indices", lambda: gtm.pixel_indices(u, v, W, H), lambda: cpp.pixel_indices(u, v, W, H)),
        ("occlusion check", lambda: gtm.occlusion_reason(cu, cv, cz, W, H, P),
         lambda: cpp.occlusion_reason(cu, cv, cz, W, H, P.mode, P.radius, P.half_width, P.rel_tol, P.abs_tol)),
        ("nearest per pixel", lambda: gtm.nearest_per_pixel(cu, cv, cz, W), lambda: cpp.nearest_per_pixel(cu, cv, cz, W, H)),
        ("WHOLE step", lambda: gtm.build_ground_truth(loaded, K, W, H, P),
         lambda: cpp.lidar_to_image(points, camera_from_source, *K, W, H, P.mode, P.radius, P.half_width,
                                    P.rel_tol, P.abs_tol, P.min_depth)),
    ]
    table = pd.DataFrame([{"stage": name, "python_ms": best_ms(py), "cpp_ms": best_ms(cc)} for name, py, cc in rows])
    table["speed_up"] = table["python_ms"] / table["cpp_ms"]
    return table.round(2)


if cpp is None:
    print("C++ module not available, nothing to time")
else:
    rng = np.random.default_rng(0)
    n = 370000                                        # about six real LiDAR sweeps
    synthetic = np.column_stack([rng.uniform(-10, 90, n), rng.uniform(-25, 25, n), rng.uniform(-1.7, 6.0, n)])
    mount = np.eye(4)
    mount[:3, :3] = [[0.0, -1.0, 0.0], [0.0, 0.0, -1.0], [1.0, 0.0, 0.0]]
    mount[:3, 3] = [0.19, 1.49, -1.68]
    print(f"synthetic frame, {n} points")
    print(stage_table(synthetic, mount).to_string(index=False))
    print("Note: the Python WHOLE step starts from camera-frame points; the C++ one also does the transform.")

synthetic frame, 370000 points
            stage  python_ms  cpp_ms  speed_up
 transform points      17.14    1.96      8.75
          project       8.58    2.18      3.94
    pixel indices      10.11    6.76      1.50
  occlusion check      66.13   40.71      1.62
nearest per pixel      75.90   13.63      5.57
       WHOLE step     100.46   73.90      1.36
Note: the Python WHOLE step starts from camera-frame points; the C++ one also does the transform.


In [6]:
# --- 6. Real data: make sure the block with LiDAR is on this runtime ---
if USE_REAL_DATA:
    from vggt_aura import aura_data as ad, inference as inf, geometry as geo, pins
    chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
    BLOCK_SCENE_IDS = ad.block_scene_ids(scene_blocks, SPLIT, BLOCK)
    started = time.time()
    if ad.block_on_disk(session.data_root, BLOCK_SCENE_IDS, LAYERS):
        print("block already on disk")
    else:
        ad.download_block(session.data_root, SPLIT, BLOCK, BLOCK_SCENE_IDS, LAYERS)
    DOWNLOAD_S = time.time() - started
    if DOWNLOAD_S < 60:                 # the block was already here, so nothing was measured just now
        DOWNLOAD_S = 450.0              # measured in `04_ground_truth`: 7.5 min for this block with LiDAR
        print("using the measurement from 04_ground_truth for download + extract: 7.5 min")
    print(f"download and extract: {DOWNLOAD_S / 60:.1f} min for {len(BLOCK_SCENE_IDS)} scenes")
else:
    print("USE_REAL_DATA is False: cells 7 and 8 are skipped")

$ /usr/bin/python3 -m fzi_aura.download /content/data/fzi-aura --revision 3404bd6b8fcd6eed53a0ec7610650a6393aabb49 --splits val --scene-ids-file /content/data/fzi-aura/_scene_ids_val_block000011.txt --layers camera_keyframes,lidar_motion_compensated_keyframes --jobs 8 --verify
download and extract: 11.6 min for 20 scenes


In [7]:
# --- 7. Real frames: does C++ give the SAME ground truth as Python, on all 40 frames of a scene? ---
if USE_REAL_DATA:       # the scene is set up even if the C++ build failed, so cell 8 still works
    import json
    from fzi_aura import FZIAURADataset

    dataset = FZIAURADataset(session.data_root, split=SPLIT)
    scene = dataset.get_scene(SCENE_ID)
    frames = inf.select_frames(scene, CAMERA)
    lidars = list(scene.available_lidars())
    calibration = scene.calibration()
    Pm = calibration.camera_projection_matrix(CAMERA)
    native = calibration.sensor(f"camera/{CAMERA}")["intrinsics"]
    K = geo.scale_intrinsics(Pm[0, 0], Pm[1, 1], Pm[0, 2], Pm[1, 2], (native["width"], native["height"]), (W, H))
    classes = json.loads((scene.path / "labels" / "semantic" / "classes.json").read_text(encoding="utf-8"))
    drop_ids = gtm.excluded_class_ids({int(c["id"]): c["name"] for c in classes["semantic_classes"]})

    report, py_s, cpp_s = [], 0.0, 0.0
    for index, frame in enumerate(frames):
        loaded = gtm.load_points_in_camera(frame, CAMERA, lidars)
        keep = ~np.isin(loaded["semantic_id"], list(drop_ids))
        t0 = time.perf_counter()
        expected = gtm.build_ground_truth(loaded, K, W, H, P, drop_ids)
        t1 = time.perf_counter()
        py_s += t1 - t0
        if cpp is None:
            continue
        got = cpp.lidar_to_image(loaded["points_cam"], np.eye(4), *K, W, H, P.mode, P.radius, P.half_width,
                                 P.rel_tol, P.abs_tol, P.min_depth, keep=keep)
        t2 = time.perf_counter()
        cpp_s += t2 - t1
        same_pixels = np.array_equal(got["u"], expected["u"]) and np.array_equal(got["v"], expected["v"])
        report.append({"frame": index, "points": len(loaded["points_cam"]), "pixels": len(expected["depth_m"]),
                       "candidates_equal": np.array_equal(got["candidate_index"], expected["candidate_index"]),
                       "verdicts_equal": np.array_equal(got["reason"], expected["reason"]), "pixels_equal": same_pixels,
                       "max_depth_diff_m": float(np.max(np.abs(got["depth_m"] - expected["depth_m"]))) if same_pixels and len(got["depth_m"]) else np.nan})
    if cpp is None:
        print(f"C++ module not available. Python alone: {py_s:.2f} s for {len(frames)} frames.")
    agreement = pd.DataFrame(report if report else [{"frame": -1, "candidates_equal": False, "verdicts_equal": False,
                                                       "pixels_equal": False, "max_depth_diff_m": np.nan}])
    print(agreement.head(5).to_string(index=False))
    print("...")
    print("frames:", len(agreement), "| all candidates equal:", bool(agreement["candidates_equal"].all()),
          "| all verdicts equal:", bool(agreement["verdicts_equal"].all()), "| all pixels equal:", bool(agreement["pixels_equal"].all()))
    print("largest depth difference over the scene: %.3g m  (the reference stores float32, so about 1e-6 is expected)" % agreement["max_depth_diff_m"].max())
    if cpp is not None:
        print(f"time for the scene: Python {py_s:.2f} s | C++ {cpp_s:.2f} s | speed-up {py_s / cpp_s:.1f}x")
        print()
        print("stage timing on the last real frame (", len(loaded["points_cam"]), "points ):")
        print(stage_table(loaded["points_cam"], np.eye(4)).to_string(index=False))

 frame  points  pixels  candidates_equal  verdicts_equal  pixels_equal  max_depth_diff_m
     0  369173   38694              True            True          True          0.000008
     1  367171   40324              True            True          True          0.000008
     2  364805   41917              True            True          True          0.000008
     3  367336   42826              True            True          True          0.000008
     4  369554   45008              True            True          True          0.000007
...
frames: 40 | all candidates equal: True | all verdicts equal: True | all pixels equal: True
largest depth difference over the scene: 1.5e-05 m  (the reference stores float32, so about 1e-6 is expected)
time for the scene: Python 3.28 s | C++ 2.40 s | speed-up 1.4x

stage timing on the last real frame ( 390437 points ):
            stage  python_ms  cpp_ms  speed_up
 transform points      20.41    4.34      4.70
          project      14.17    1.68      8.43


In [8]:
# --- 8. Where does the time of ONE scene actually go? (this decided where the speed-ups went) ---
if USE_REAL_DATA:
    from vggt_aura import evaluation as ev, objects as ob

    def timed(fn):
        started = time.perf_counter()
        value = fn()
        return value, time.perf_counter() - started

    frame = frames[0]
    _, read_pcd_s = timed(lambda: [f.load_lidar(l, stage=gtm.STAGE) for f in frames for l in lidars])
    _, read_labels_s = timed(lambda: [f.load_semantics(l) for f in frames for l in lidars])
    loaded_all, load_all_s = timed(lambda: [gtm.load_points_in_camera(f, CAMERA, lidars) for f in frames])
    truth, project_py_s = timed(lambda: [gtm.build_ground_truth(x, K, W, H, P, drop_ids) for x in loaded_all])
    labels, labels_s = timed(lambda: ob.scene_motion_labels(frames, CAMERA, truth))
    steps = [("download + extract, share of one scene", DOWNLOAD_S / len(BLOCK_SCENE_IDS)),
             ("read 240 LiDAR files", read_pcd_s), ("read 240 label files", read_labels_s),
             ("project + occlusion, Python", project_py_s), ("moving-object labels from boxes", labels_s)]
    if cpp is not None:
        steps.append(("project + occlusion, C++ (would replace the Python line)", cpp_s))
    npz_path, json_path = inf.prediction_paths(session.persist_root, scene.name, CAMERA, pins.CHECKPOINT_PUBLIC_512)
    if npz_path.is_file():
        arrays, meta = inf.load_predictions(npz_path, json_path)
        class_names = {int(c["id"]): c["name"] for c in classes["semantic_classes"]}
        _, score_s = timed(lambda: ev.evaluate_scene(SCENE_ID, arrays, truth, labels, class_names, K[0], K[1]))
        steps += [("score the scene (5 protocols, all strata)", score_s), ("model inference (measured on the A100 in `05a_one_block_predict`)", meta.get("forward_seconds", np.nan))]
    breakdown = pd.DataFrame(steps, columns=["step", "seconds"])
    counted = ~breakdown["step"].str.contains("C\\+\\+")
    breakdown["percent"] = np.where(counted, 100 * breakdown["seconds"] / breakdown.loc[counted, "seconds"].sum(), np.nan)
    print(breakdown.round(2).to_string(index=False))
    total = breakdown.loc[counted, "seconds"].sum()
    print()
    print(f"one scene end to end: {total:.0f} s | 100 scenes: {100 * total / 3600:.1f} h | 1000 scenes: {1000 * total / 3600:.1f} h")
    ad.save_json(session.persist_root / "metrics" / "phase6_timing.json",
                 {"steps": breakdown.to_dict("records"), "scene": SCENE_ID, "download_block_s": DOWNLOAD_S})

                                                    step  seconds  percent
                  download + extract, share of one scene    34.89    66.35
                                    read 240 LiDAR files     0.76     1.45
                                    read 240 label files     0.20     0.37
                             project + occlusion, Python     2.37     4.51
                         moving-object labels from boxes     3.43     6.52
project + occlusion, C++ (would replace the Python line)     2.40      NaN
               score the scene (5 protocols, all strata)     7.96    15.13
           model inference (measured on the A100 in 05a)     2.98     5.67

one scene end to end: 53 s | 100 scenes: 1.5 h | 1000 scenes: 14.6 h
